In [ ]:
# Inference throughput on Kaggle's T4 x2 machine (same image as knee-submit), on N training studies posing as a test set.
# Question: does knee-submit's 8 h budget cut our ensembles on the hidden test, and do the CPUs or the GPU limit it?
import glob, os, subprocess, sys, time, threading
T0 = time.time()
def find(pattern):
    return next(p for depth in range(4) for p in glob.glob('/kaggle/input/' + '*/' * depth + pattern))
KNEE = os.path.dirname(os.path.dirname(find('knee/infer.py')))
DATA = os.path.dirname(find('train_series.csv'))
print(KNEE, DATA)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', f'{KNEE}/wheels',
                'pylibjpeg', 'pylibjpeg-libjpeg', 'pylibjpeg-openjpeg', 'python-gdcm'], check=False)
sys.path.insert(0, KNEE)
import numpy as np, pandas as pd, torch
from pathlib import Path
print('GPUs', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], 'CPUs', os.cpu_count(),
      'sched CPUs', len(os.sched_getaffinity(0)))
subprocess.run(['nproc'])
subprocess.run(['free', '-g'])

In [ ]:
N = 60
ts = pd.read_csv(f'{DATA}/train_series.csv')
studies = ts['StudyInstanceUID'].drop_duplicates().sample(N, random_state=0)
fake = Path('/kaggle/working/fake'); fake.mkdir(exist_ok=True)
ts[ts['StudyInstanceUID'].isin(studies)].to_csv(fake / 'test_series.csv', index=False)
pd.DataFrame({'StudyInstanceUID': studies.to_numpy()}).to_csv(fake / 'test.csv', index=False)
if not (fake / 'test_series').exists():
    os.symlink(f'{DATA}/train_series', fake / 'test_series')
from knee import infer
from knee.data import StudyDataset, assign_slots
from torch.utils.data import DataLoader
W = os.cpu_count()
t = time.time()
series = infer.prepare_cache(fake, Path('/tmp/knee_cache'), W, sizes=[256, 384])
dec = time.time() - t
print(f'DECODE {N} studies, {len(series)} series: {dec:.0f}s = {dec / N:.2f} s/study (both sizes)')
slots = assign_slots(series).set_index('StudyInstanceUID').reindex(studies.to_numpy()).reset_index()
M = {os.path.basename(p).split('_')[2]: Path(p) for p in glob.glob(f'{KNEE}/models/*')}
print(sorted(M))

In [ ]:
import json
def cfg(m): return json.loads((M[m] / 'config.json').read_text())
def timed(m, dev, windows, workers):
    c = cfg(m); cd = infer.size_dir(Path('/tmp/knee_cache'), c.get('cache_size', 256))
    t = time.time(); infer.predict_model(M[m], slots, cd, torch.device(dev), workers, windows); return time.time() - t
def loader_only(m, windows, workers):
    c = cfg(m); cd = infer.size_dir(Path('/tmp/knee_cache'), c.get('cache_size', 256))
    ds = StudyDataset(slots, n_windows=windows, img_size=c['img_size'], span=tuple(c.get('span', (0.08, 0.92))),
                      zoom=c.get('zoom', 1.0), cache_dir=cd)
    t = time.time()
    for b in DataLoader(ds, batch_size=max(1, 48 // windows), num_workers=workers): pass
    return time.time() - t
timed('e41', 'cuda:0', 24, W)  # warm-up (cudnn, page cache)
R = {}
for m in ['e46', 'e41', 'e43', 'e37', 'e33']:
    if m not in M: continue
    for w in [24, 12]:
        R[(m, w)] = timed(m, 'cuda:0', w, W)
        print(f'MODEL {m} img {cfg(m)["img_size"]} windows {w}: {R[(m, w)]:.0f}s = {R[(m, w)] / N:.2f} s/study (5 folds, 1 GPU)', flush=True)
for m in ['e46', 'e41']:
    for w in [24, 12]:
        t = loader_only(m, w, W)
        print(f'LOADER {m} windows {w}: {t:.0f}s = {t / N:.2f} s/study ({W} workers, no GPU)', flush=True)

In [ ]:
# two models at once, one per T4, each with half the CPU workers
def run2(a, b, w):
    out = {}
    th = [threading.Thread(target=lambda k, d: out.__setitem__(k, timed(k, d, w, max(1, W // 2))), args=(k, d))
          for k, d in [(a, 'cuda:0'), (b, 'cuda:1')]]
    t = time.time(); [x.start() for x in th]; [x.join() for x in th]
    return time.time() - t, out
if torch.cuda.device_count() > 1:
    for a, b, w in [('e46', 'e53', 24), ('e41', 'e38', 24)]:
        wall, out = run2(a, b, w)
        print(f'TWO GPUS {a}+{b} windows {w}: wall {wall:.0f}s vs sequential {R.get((a, w), float("nan")) * 2:.0f}s (est.), per model {out}', flush=True)
print(f'total {(time.time() - T0) / 60:.1f} min')